# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # TODO: Implement a tokenization strategy
    b, c, h, w = x.shape
    p = patch_size
    if h % p != 0 or w % p != 0:
        raise ValueError("image dimensions must be divisible by patch_size")
    # (B, C, H, W) -> (B,C,H/P, W/P, P, P)
    patches = x.unfold(2, p, p).unfold(3, p, p)
    # (B, C, H/P, W/P, P, P) -> (B, H/P, W/P, C, P, P)
    patches = patches.permute(0, 2, 3, 1, 4, 5)
    return patches.reshape(b, (h // p) * (w // p), c * p * p)

In [3]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.projection = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.projection(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.pos_embed = nn.Parameter(torch.empty(1,num_tokens,d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        if x.ndim != 3 or x.shape[1:] != self.pos_embed.shape[1:]:
            raise ValueError("token count and feature dimension must match position embeddings")
        return x + self.pos_embed

In [4]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=False)
        self.fc2 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        hidden = self.dropout(F.gelu(self.fc1(x)))
        return self.dropout(self.fc2(hidden))

class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        # TODO: implement
        self.variant = variant.lower()
        self.gate = nn.Linear(d_model, d_ff_gated, bias=False)
        self.value = nn.Linear(d_model, d_ff_gated, bias=False)
        self.output = nn.Linear(d_ff_gated, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        a = self.gate(x)
        b = self.value(x)
        activated = F.gelu(a) if self.variant == "geglu" else F.silu(a)
        hidden = self.dropout(activated * b)
        return self.dropout(self.output(hidden))


In [5]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        # TODO: implement. For attention use nn.MultiHeadAttention 
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.attention_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = mlp

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        normalized = self.norm1(x)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        x = x + self.attention_dropout(attended)
        return x + self.mlp(self.norm2(x))

In [6]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        # TODO: implement a strategy for embedding the patches
        self.patch_embed = PatchEmbed(patch_dim, d_model)
        self.pos_embed = PositionalEmbedding(self.num_tokens, d_model)
        self.embed_dropout = nn.Dropout(dropout)

        # TODO: implement a strategy to select the right mlp version for your experiment
        kind = mlp_kind.lower()
        gated_width = max(1, (2 * d_ff) // 3)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=(
                    FeedForward(d_model, d_ff, dropout)
                    if kind == "gelu"
                    else GLUFeedForward(d_model, gated_width, dropout, kind)
                ), # TODO: Feed your mlp to the encoder blocks
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # TODO: Add a head to project to the amount of output classes you have
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement
        tokens = self.patch_embed(patchify(x, self.patch_size))
        tokens = self.embed_dropout(self.pos_embed(tokens))
        for block in self.blocks:
            tokens = block(tokens)
        pooled = self.norm(tokens).mean(dim=1)
        return self.head(pooled)


In [7]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [8]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    torch.manual_seed(cfg.seed)
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    train_epoch_losses: list[float] = []
    test_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        train_loss_sum = 0.0
        train_total = 0
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits,yb) # TODO: Your criterion
            loss.backward()
            opt.step()

            train_losses.append(loss.item())
            n = yb.numel()
            train_loss_sum += loss.item() * n
            train_total += n

        train_epoch_losses.append(train_loss_sum / train_total)
        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        test_loss_sum = 0.0

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    return {
        # TODO: Return your metrics that you think will support your claim for this experiment
        "mlp_kind": mlp_kind,
        "seed": cfg.seed,
        "num_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "train_losses": train_losses,
        "train_epoch_losses": train_epoch_losses,
        "test_accs": test_accs,
        "final_test_accuracy": test_accs[-1],
        "best_test_accuracy": max(test_accs),
        "best_epoch": test_accs.index(max(test_accs)) + 1,
    }

In [9]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="mps")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

# Compare the GELU baseline with GELU and SiLU gating.
runs = ["gelu", "geglu", "swiglu"]
results = []

for kind in runs:
    torch.manual_seed(cfg.seed)
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nRun: {kind} | parameters: {num_params:,}")
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)

print("\nResults:")
for result in results:
    print(f"\n{result['mlp_kind']} | parameters: {result['num_parameters']:,}")
    for epoch, (loss, acc) in enumerate(
        zip(result["train_epoch_losses"], result["test_accs"]), start=1
    ):
        print(f"  epoch {epoch}: train loss = {loss:.4f}, test accuracy = {acc:.2%}")
    print(f"  final accuracy: {result['final_test_accuracy']:.2%}")
    print(f"  best accuracy: {result['best_test_accuracy']:.2%} "
          f"(epoch {result['best_epoch']})")



Run: gelu | parameters: 104,330
[gelu] epoch 1/5 | test acc: 0.8184
[gelu] epoch 2/5 | test acc: 0.9065
[gelu] epoch 3/5 | test acc: 0.9320
[gelu] epoch 4/5 | test acc: 0.9436
[gelu] epoch 5/5 | test acc: 0.9484

Run: geglu | parameters: 104,074
[geglu] epoch 1/5 | test acc: 0.8369
[geglu] epoch 2/5 | test acc: 0.9239
[geglu] epoch 3/5 | test acc: 0.9495
[geglu] epoch 4/5 | test acc: 0.9581
[geglu] epoch 5/5 | test acc: 0.9640

Run: swiglu | parameters: 104,074
[swiglu] epoch 1/5 | test acc: 0.8297
[swiglu] epoch 2/5 | test acc: 0.9218
[swiglu] epoch 3/5 | test acc: 0.9496
[swiglu] epoch 4/5 | test acc: 0.9603
[swiglu] epoch 5/5 | test acc: 0.9641

Results:

gelu | parameters: 104,330
  epoch 1: train loss = 1.2681, test accuracy = 81.84%
  epoch 2: train loss = 0.4946, test accuracy = 90.65%
  epoch 3: train loss = 0.3356, test accuracy = 93.20%
  epoch 4: train loss = 0.2602, test accuracy = 94.36%
  epoch 5: train loss = 0.2153, test accuracy = 94.84%
  final accuracy: 94.84%
  bes